# HR-GPT — training notebook (Kaggle & Colab)

Fine-tune **GPT-2** to read an employee description and predict four things at once:
**attrition** (will they leave?), **job level**, **job satisfaction** and **monthly income**.

The notebook is **self-contained**: all the code is inside, identical to the
[HR-GPT repo](https://github.com/benisalla/HR-GPT), with the same defaults. Upload it and
press **Run all**.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4* | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* (data + GPT-2 weights) | on by default |
| Outputs kept in | `/kaggle/working/hr-gpt` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

It also runs on a CPU, but slowly: there, set `INIT = "distilgpt2"`.

**Before the long run:** set `SMOKE_TEST = True` in section 2.2 and *Run all* once
(about a minute): it checks the whole notebook with a tiny model. Then set it back to
`False` and run again.

**Sections:** 1. Setup · 2. Config · 3. Data · 4. Model (how the architecture works) ·
5. Training (live dashboard) · 6. Test results vs baselines · 7. Try the model · 8. Download

## 1. Setup

Kaggle and Colab already have everything we need (PyTorch, transformers, pandas,
scikit-learn, matplotlib). Nothing to install.

In [ ]:
import hashlib
import json
import math
import random
import shutil
import time
import urllib.request
from collections.abc import Callable, Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, clear_output, display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import Tensor, nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("torch", torch.__version__, "| GPU:", gpu)

## 2. Config

### 2.1 Config classes



*Same code as `src/hr_gpt/config.py`.*

Every setting in one place: the tasks, the data, the model and the training run.

In [ ]:
CLASSIFICATION = "classification"


REGRESSION = "regression"


PRETRAINED = ("gpt2", "distilgpt2")


@dataclass(frozen=True)
class TaskSpec:
    """One prediction head."""

    column: str  # CSV column with the target
    kind: str  # CLASSIFICATION or REGRESSION
    classes: tuple[str, ...] = ()  # class names, in label order (classification)
    values: tuple = ()  # the CSV value of each class, same order (label = position)
    loss_weight: float = 1.0  # weight of this task in the total loss
    balanced: bool = False  # weight classes by 1 / frequency (for rare classes)
    log_target: bool = False  # regression: predict log(value) (for skewed values)

    @property
    def n_outputs(self) -> int:
        return len(self.classes) if self.kind == CLASSIFICATION else 1


TASKS: dict[str, TaskSpec] = {
    # 16% of employees leave: balanced weights stop the head from always saying "stays"
    "Attrition": TaskSpec(
        "Attrition",
        CLASSIFICATION,
        classes=("Stays", "Leaves"),
        values=("No", "Yes"),
        balanced=True,
    ),
    "JobLevel": TaskSpec(
        "JobLevel",
        CLASSIFICATION,
        classes=("Level 1", "Level 2", "Level 3", "Level 4", "Level 5"),
        values=(1, 2, 3, 4, 5),
    ),
    "JobSatisfaction": TaskSpec(
        "JobSatisfaction",
        CLASSIFICATION,
        classes=("Low", "Medium", "High", "Very High"),
        values=(1, 2, 3, 4),
    ),
    # Incomes range from 1k to 20k and are right-skewed: learn log(income), standardised
    "MonthlyIncome": TaskSpec("MonthlyIncome", REGRESSION, log_target=True),
}


@dataclass
class DataConfig:
    """Where the data lives and how it is split."""

    data_dir: Path = Path("data")
    csv_url: str = (
        "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/"
        "master/data/emp_attrition.csv"
    )
    csv_md5: str = "ad8207459e5732574372cf8ff619883f"
    val_fraction: float = 0.1
    test_fraction: float = 0.1
    split_seed: int = 1337
    max_tokens: int = 512  # longest sentence is ~220 GPT-2 tokens

    @property
    def raw_csv(self) -> Path:
        return Path(self.data_dir) / "raw" / "hr_employee_attrition.csv"

    @property
    def processed_dir(self) -> Path:
        return Path(self.data_dir) / "processed"


@dataclass
class ModelConfig:
    """Transformer size. With `init` = gpt2/distilgpt2 the size comes from that checkpoint."""

    init: str = "gpt2"  # "gpt2" | "distilgpt2" | "scratch"
    vocab_size: int = 50258  # GPT-2's 50257 + <|pad|>
    block_size: int = 1024
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    head_dropout: float = 0.1
    bias: bool = True

    def validate(self) -> "ModelConfig":
        if self.init not in (*PRETRAINED, "scratch"):
            raise ValueError(f"init must be one of {(*PRETRAINED, 'scratch')}, got {self.init!r}")
        if self.n_embd % self.n_head != 0:
            raise ValueError(f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})")
        if not 0 <= self.dropout < 1 or not 0 <= self.head_dropout < 1:
            raise ValueError("dropout values must be in [0, 1)")
        return self


@dataclass
class TrainConfig:
    """How long and how fast to train."""

    output_dir: Path = Path("checkpoints/run")
    epochs: int = 8
    batch_size: int = 16
    lr: float = 3e-5  # backbone (pretrained, so small)
    head_lr: float = 1e-3  # task heads (new, so larger)
    weight_decay: float = 0.01
    warmup_fraction: float = 0.1
    grad_clip: float = 1.0
    freeze_embeddings: bool = True  # keep GPT-2's 50k-token embedding table fixed (38.6M params)
    patience: int = 3  # stop after this many epochs without a better val loss (0 = never)
    seed: int = 1337
    device: str = "auto"  # "auto" | "cpu" | "cuda" | "mps"
    precision: str = "auto"  # "auto" (bf16 on supported GPUs, else fp32) | "fp32" | "bf16"
    log_every: int = 20  # steps
    num_workers: int = 0
    limit: int | None = None  # use only the first N training rows (quick checks)

    def validate(self) -> "TrainConfig":
        if self.epochs <= 0 or self.batch_size <= 0:
            raise ValueError("epochs and batch_size must be > 0")
        if self.lr <= 0 or self.head_lr <= 0:
            raise ValueError("learning rates must be > 0")
        if not 0 <= self.warmup_fraction < 1:
            raise ValueError("warmup_fraction must be in [0, 1)")
        if self.precision not in ("auto", "fp32", "bf16"):
            raise ValueError(f"precision must be auto, fp32 or bf16, got {self.precision!r}")
        return self


def to_jsonable(obj: Any) -> Any:
    """Dataclass -> dict with Paths as strings (for checkpoints and JSON files)."""
    data = asdict(obj) if hasattr(obj, "__dataclass_fields__") else obj
    if isinstance(data, dict):
        return {k: to_jsonable(v) for k, v in data.items()}
    if isinstance(data, (list, tuple)):
        return [to_jsonable(v) for v in data]
    if isinstance(data, Path):
        return str(data)
    return data


def tasks_from_dict(data: dict[str, dict[str, Any]]) -> dict[str, TaskSpec]:
    return {
        name: TaskSpec(
            **{**spec, "classes": tuple(spec["classes"]), "values": tuple(spec["values"])}
        )
        for name, spec in data.items()
    }

### 2.2 Settings for this run

The defaults are the repo's `hr-train` defaults. `SMOKE_TEST = True` runs the whole
notebook in about a minute with a tiny random model (to check that everything works).

In [ ]:
SMOKE_TEST = False  # True = tiny quick check of the whole notebook
USE_GOOGLE_DRIVE = False  # Colab only: keep outputs on Google Drive
INIT = "gpt2"  # "gpt2" (12 layers) | "distilgpt2" (6 layers, 2x faster) | "scratch"
EPOCHS = 8

if Path("/kaggle/working").exists():
    BASE = Path("/kaggle/working/hr-gpt")
elif USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/hr-gpt")
else:
    BASE = Path("hr-gpt")

data_config = DataConfig(data_dir=BASE / "data")
model_config = ModelConfig(init=INIT)
train_config = TrainConfig(output_dir=BASE / "run", epochs=EPOCHS)
if SMOKE_TEST:
    model_config = ModelConfig(init="scratch", n_layer=2, n_head=2, n_embd=64)
    train_config = TrainConfig(output_dir=BASE / "smoke", epochs=2, limit=96, lr=1e-3, log_every=2)
print(model_config, train_config, sep="\n")

## 3. Data

### 3.1 The employee record

Each employee has 27 input columns. They are written as one English sentence, which is what GPT-2 is good at reading. The four columns we predict are never in the sentence.

*Same code as `src/hr_gpt/features.py`.*

The employee record: which columns exist, how they look in the UI, how they read as text.

```text
This file is the single source of truth for the data schema. Training, evaluation,
the command-line predictor and the web app all use it, so a record is always turned
into exactly the same sentence everywhere.

Values are kept as they appear in the IBM CSV (e.g. Education = 1..5,
BusinessTravel = "Travel_Rarely"); only the sentence uses human words.
```

In [ ]:
LEVEL4 = {1: "Low", 2: "Medium", 3: "High", 4: "Very High"}


EDUCATION = {1: "Below College", 2: "College", 3: "Bachelor", 4: "Master", 5: "Doctor"}


PERFORMANCE = {1: "Low", 2: "Good", 3: "Excellent", 4: "Outstanding"}


WORK_LIFE = {1: "Bad", 2: "Good", 3: "Better", 4: "Best"}


@dataclass(frozen=True)
class Feature:
    """One input column."""

    name: str  # column name in the CSV
    label: str  # label in the web app
    group: str  # tab in the web app
    phrase: Callable[[Any], str]  # value -> words, e.g. 41 -> "is 41 years old"
    default: Any  # value shown in the web app before anything is loaded
    choices: dict[Any, str] = field(default_factory=dict)  # categorical: CSV value -> label
    minimum: float | None = None  # numeric: allowed range in the web app
    maximum: float | None = None

    @property
    def is_categorical(self) -> bool:
        return bool(self.choices)


def _plural(n: int, word: str) -> str:
    return f"{n} {word}" if n == 1 else f"{n} {word}s"


def _travel(value: str) -> str:
    return {
        "Non-Travel": "does not travel for work",
        "Travel_Rarely": "travels rarely for work",
        "Travel_Frequently": "travels frequently for work",
    }[value]


def _num(name: str, label: str, group: str, template: str, default: int, lo: int, hi: int):
    return Feature(
        name, label, group, lambda v: template.format(int(v)), default, minimum=lo, maximum=hi
    )


def _years(name: str, label: str, group: str, template: str, default: int, hi: int):
    return Feature(
        name,
        label,
        group,
        lambda v: template.format(_plural(int(v), "year")),
        default,
        minimum=0,
        maximum=hi,
    )


def _scale(name: str, label: str, group: str, template: str, scale: dict, default: int):
    return Feature(
        name,
        label,
        group,
        lambda v: template.format(scale[int(v)].lower()),
        default,
        choices=dict(scale),
    )


def _cat(name: str, label: str, group: str, template: str, values: list[str], default: str):
    return Feature(
        name,
        label,
        group,
        lambda v: template.format(v.lower()),
        default,
        choices={v: v for v in values},
    )


PERSONAL, JOB, PAY, SURVEY = "Personal", "Job", "Pay", "Survey"


FEATURES: list[Feature] = [
    _num("Age", "Age", PERSONAL, "is {} years old", 35, 18, 60),
    Feature(
        "Gender",
        "Gender",
        PERSONAL,
        lambda v: f"is {v.lower()}",
        "Male",
        choices={"Male": "Male", "Female": "Female"},
    ),
    _cat(
        "MaritalStatus",
        "Marital status",
        PERSONAL,
        "is {}",
        ["Single", "Married", "Divorced"],
        "Married",
    ),
    _scale("Education", "Education", PERSONAL, "has {} education", EDUCATION, 3),
    _cat(
        "EducationField",
        "Education field",
        PERSONAL,
        "studied {}",
        ["Life Sciences", "Medical", "Marketing", "Technical Degree", "Human Resources", "Other"],
        "Life Sciences",
    ),
    _num("DistanceFromHome", "Distance from home", PERSONAL, "lives {} km from work", 7, 1, 29),
    _cat(
        "Department",
        "Department",
        JOB,
        "works in the {} department",
        ["Research & Development", "Sales", "Human Resources"],
        "Research & Development",
    ),
    _cat(
        "JobRole",
        "Job role",
        JOB,
        "works as a {}",
        [
            "Sales Executive",
            "Research Scientist",
            "Laboratory Technician",
            "Manufacturing Director",
            "Healthcare Representative",
            "Manager",
            "Sales Representative",
            "Research Director",
            "Human Resources",
        ],
        "Research Scientist",
    ),
    Feature(
        "BusinessTravel",
        "Business travel",
        JOB,
        _travel,
        "Travel_Rarely",
        choices={
            "Non-Travel": "Never",
            "Travel_Rarely": "Rarely",
            "Travel_Frequently": "Frequently",
        },
    ),
    Feature(
        "OverTime",
        "Works overtime",
        JOB,
        lambda v: "works overtime" if v == "Yes" else "does not work overtime",
        "No",
        choices={"No": "No", "Yes": "Yes"},
    ),
    Feature(
        "NumCompaniesWorked",
        "Previous companies",
        JOB,
        lambda v: f"has worked at {int(v)} {'company' if int(v) == 1 else 'companies'} before",
        2,
        minimum=0,
        maximum=9,
    ),
    _years("TotalWorkingYears", "Total working years", JOB, "has {} of experience", 10, 40),
    _years("YearsAtCompany", "Years at company", JOB, "has spent {} at the company", 5, 40),
    _years("YearsInCurrentRole", "Years in current role", JOB, "has {} in the current role", 3, 18),
    _years(
        "YearsSinceLastPromotion",
        "Years since last promotion",
        JOB,
        "was last promoted {} ago",
        1,
        15,
    ),
    _years(
        "YearsWithCurrManager", "Years with manager", JOB, "has {} with the current manager", 3, 17
    ),
    _num(
        "TrainingTimesLastYear",
        "Trainings last year",
        JOB,
        "attended {} trainings last year",
        3,
        0,
        6,
    ),
    _num("DailyRate", "Daily rate", PAY, "has a daily rate of {}", 800, 100, 1500),
    _num("HourlyRate", "Hourly rate", PAY, "has an hourly rate of {}", 65, 30, 100),
    _num("MonthlyRate", "Monthly rate", PAY, "has a monthly rate of {}", 14000, 2000, 27000),
    _num("PercentSalaryHike", "Last salary hike (%)", PAY, "received a {}% raise", 14, 11, 25),
    _num("StockOptionLevel", "Stock option level", PAY, "has stock option level {}", 1, 0, 3),
    _scale(
        "PerformanceRating",
        "Performance rating",
        SURVEY,
        "has a performance rating of {}",
        PERFORMANCE,
        3,
    ),
    _scale("JobInvolvement", "Job involvement", SURVEY, "shows {} job involvement", LEVEL4, 3),
    _scale(
        "EnvironmentSatisfaction",
        "Environment satisfaction",
        SURVEY,
        "reports {} environment satisfaction",
        LEVEL4,
        3,
    ),
    _scale(
        "RelationshipSatisfaction",
        "Relationship satisfaction",
        SURVEY,
        "reports {} relationship satisfaction",
        LEVEL4,
        3,
    ),
    _scale(
        "WorkLifeBalance", "Work-life balance", SURVEY, "has {} work-life balance", WORK_LIFE, 3
    ),
]


FEATURES_BY_NAME: dict[str, Feature] = {f.name: f for f in FEATURES}


TARGET_COLUMNS = ["Attrition", "JobLevel", "JobSatisfaction", "MonthlyIncome"]


DROPPED_COLUMNS = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]


ALL_COLUMNS = [f.name for f in FEATURES] + TARGET_COLUMNS


def default_record() -> dict[str, Any]:
    """A plausible employee, used as the starting point in the web app."""
    return {f.name: f.default for f in FEATURES}


def validate_record(record: dict[str, Any]) -> dict[str, Any]:
    """Check a record and return it with numbers as int. Raises ValueError with a clear message."""
    clean: dict[str, Any] = {}
    for feature in FEATURES:
        if feature.name not in record or record[feature.name] is None:
            raise ValueError(f"missing value for '{feature.name}'")
        value = record[feature.name]
        if feature.is_categorical:
            if isinstance(value, float) and value.is_integer():
                value = int(value)
            if value not in feature.choices:
                allowed = ", ".join(map(str, feature.choices))
                raise ValueError(f"'{feature.name}' must be one of: {allowed} (got {value!r})")
        else:
            try:
                value = round(float(value))
            except (TypeError, ValueError):
                raise ValueError(f"'{feature.name}' must be a number (got {value!r})") from None
            if value < 0:
                raise ValueError(f"'{feature.name}' cannot be negative (got {value})")
        clean[feature.name] = value
    return clean


def record_to_text(record: dict[str, Any]) -> str:
    """Turn an employee record into the sentence the model reads.

    >>> record_to_text(default_record())[:60]
    'The employee is 35 years old, is male, is married, has bache'
    """
    record = validate_record(record)
    phrases = [f.phrase(record[f.name]) for f in FEATURES]
    return "The employee " + ", ".join(phrases) + "."

### 3.2 Download, split, tokenize, batch



*Same code as `src/hr_gpt/data.py`.*

Data pipeline: download the CSV, split it, turn each employee into tokens and targets.

```text
raw CSV --prepare_data--> data/processed/{train,val,test}.csv
row --record_to_text--> "The employee is 41 years old, ..." --tokenizer--> ids + <eos>
row --TargetEncoder--> {"Attrition": 1, "JobLevel": 1, ..., "MonthlyIncome": 0.37}
```

In [ ]:
PAD_TOKEN = "<|pad|>"


SPLITS = ("train", "val", "test")


def download_csv(config: DataConfig) -> Path:
    """Download the IBM HR Analytics CSV once (1470 rows, ~230 KB) and check its MD5."""
    path = config.raw_csv
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"downloading {config.csv_url}")
        tmp = path.with_suffix(".part")
        urllib.request.urlretrieve(config.csv_url, tmp)
        tmp.rename(path)
    digest = hashlib.md5(path.read_bytes()).hexdigest()
    if config.csv_md5 and digest != config.csv_md5:
        raise ValueError(f"{path} has MD5 {digest}, expected {config.csv_md5}; delete it and retry")
    return path


def clean_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Keep the known columns, strip text, check there are no missing values."""
    df = df.rename(columns=lambda c: c.strip().lstrip("﻿"))
    missing = [c for c in ALL_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing columns: {missing}")
    df = df.drop(columns=[c for c in DROPPED_COLUMNS if c in df.columns])[ALL_COLUMNS].copy()
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(str).str.strip()
    if df.isna().any().any():
        raise ValueError(f"missing values in: {df.columns[df.isna().any()].tolist()}")
    return df


def stratified_split(
    df: pd.DataFrame, column: str, val_fraction: float, test_fraction: float, seed: int
) -> dict[str, pd.DataFrame]:
    """Shuffle and split so each split keeps the same share of each `column` value.

    Attrition is rare (16%), so a plain random split can leave val/test with too
    few leavers to measure anything.
    """
    rng = np.random.default_rng(seed)
    parts: dict[str, list[np.ndarray]] = {name: [] for name in SPLITS}
    for _, group in df.groupby(column, sort=True):
        idx = rng.permutation(group.index.to_numpy())
        n_val = round(len(idx) * val_fraction)
        n_test = round(len(idx) * test_fraction)
        parts["test"].append(idx[:n_test])
        parts["val"].append(idx[n_test : n_test + n_val])
        parts["train"].append(idx[n_test + n_val :])
    return {
        name: df.loc[rng.permutation(np.concatenate(chunks))].reset_index(drop=True)
        for name, chunks in parts.items()
    }


def prepare_data(config: DataConfig, force: bool = False) -> dict[str, Path]:
    """Download (if needed), clean and split the data. Returns the path of each split."""
    out = config.processed_dir
    paths = {name: out / f"{name}.csv" for name in SPLITS}
    if not force and all(p.exists() for p in paths.values()):
        return paths
    df = clean_frame(pd.read_csv(download_csv(config)))
    splits = stratified_split(
        df, "Attrition", config.val_fraction, config.test_fraction, config.split_seed
    )
    out.mkdir(parents=True, exist_ok=True)
    stats = {}
    for name, frame in splits.items():
        frame.to_csv(paths[name], index=False)
        stats[name] = {
            "rows": len(frame),
            "attrition_rate": round(float((frame["Attrition"] == "Yes").mean()), 4),
        }
    (out / "split_stats.json").write_text(json.dumps(stats, indent=2))
    print("prepared:", ", ".join(f"{n}={s['rows']} rows" for n, s in stats.items()))
    return paths


def load_split(config: DataConfig, split: str) -> pd.DataFrame:
    if split not in SPLITS:
        raise ValueError(f"split must be one of {SPLITS}, got {split!r}")
    return pd.read_csv(prepare_data(config)[split])


def frame_to_records(df: pd.DataFrame) -> list[dict[str, Any]]:
    """DataFrame -> list of dicts with plain Python values (numpy ints become int)."""
    return [
        {k: (v.item() if hasattr(v, "item") else v) for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ]


@dataclass
class RegressionScale:
    """Standardise a regression target: z = (f(y) - mean) / std, f = log or identity."""

    mean: float
    std: float
    log: bool

    def encode(self, value: float) -> float:
        x = math.log(value) if self.log else float(value)
        return (x - self.mean) / self.std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        x = z * self.std + self.mean
        return torch.exp(x) if self.log else x


class TargetEncoder:
    """Turns raw CSV targets into training labels, and model outputs back into values.

    Regression scales are fitted on the training split only, then stored in the
    checkpoint so inference uses exactly the same numbers.
    """

    def __init__(self, tasks: dict[str, TaskSpec], scales: dict[str, RegressionScale]):
        self.tasks = tasks
        self.scales = scales

    @classmethod
    def fit(cls, tasks: dict[str, TaskSpec], train: pd.DataFrame) -> "TargetEncoder":
        scales = {}
        for name, spec in tasks.items():
            if spec.kind != CLASSIFICATION:
                values = train[spec.column].astype(float).to_numpy()
                x = np.log(values) if spec.log_target else values
                scales[name] = RegressionScale(float(x.mean()), float(x.std()), spec.log_target)
        return cls(tasks, scales)

    def encode(self, record: dict[str, Any]) -> dict[str, float | int]:
        targets: dict[str, float | int] = {}
        for name, spec in self.tasks.items():
            value = record[spec.column]
            if spec.kind == CLASSIFICATION:
                if value not in spec.values:
                    raise ValueError(f"{name}: unknown value {value!r}, expected {spec.values}")
                targets[name] = spec.values.index(value)
            else:
                targets[name] = self.scales[name].encode(value)
        return targets

    def class_weights(self, name: str, train: pd.DataFrame) -> torch.Tensor | None:
        """1 / frequency weights (mean 1) for a `balanced` task, else None."""
        spec = self.tasks[name]
        if spec.kind != CLASSIFICATION or not spec.balanced:
            return None
        counts = np.array([(train[spec.column] == v).sum() for v in spec.values], dtype=float)
        weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
        return torch.tensor(weights, dtype=torch.float32)

    def state_dict(self) -> dict[str, Any]:
        return {name: vars(scale) for name, scale in self.scales.items()}

    @classmethod
    def from_state_dict(cls, tasks: dict[str, TaskSpec], state: dict[str, Any]) -> "TargetEncoder":
        return cls(tasks, {name: RegressionScale(**s) for name, s in state.items()})


def build_tokenizer(name_or_path: str = "gpt2"):
    """GPT-2 BPE tokenizer with an extra <|pad|> token (id 50257)."""
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(name_or_path)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": PAD_TOKEN})
    return tokenizer


def encode_text(tokenizer, text: str, max_tokens: int) -> list[int]:
    """text -> token ids + <eos>. The model reads the <eos> position as the summary."""
    ids = tokenizer.encode(text, add_special_tokens=False)[: max_tokens - 1]
    return ids + [tokenizer.eos_token_id]


class HRDataset(Dataset):
    """One item per employee: token ids of its sentence + its targets for every task."""

    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
        targets: TargetEncoder,
        max_tokens: int,
    ):
        self.records = frame_to_records(frame)
        self.texts = [record_to_text(r) for r in self.records]
        self.ids = [encode_text(tokenizer, t, max_tokens) for t in self.texts]
        self.targets = [targets.encode(r) for r in self.records]

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        return {"input_ids": self.ids[index], "targets": self.targets[index]}


@dataclass
class Collator:
    """Pad a list of items to the longest sentence; padding goes at the end.

    The model is causal, so padding after the last real token cannot change it,
    and `lengths` says where that last token is.
    """

    pad_token_id: int

    def __call__(self, items: Sequence[dict[str, Any]]) -> dict[str, Any]:
        lengths = torch.tensor([len(it["input_ids"]) for it in items])
        input_ids = torch.full((len(items), int(lengths.max())), self.pad_token_id)
        for row, item in enumerate(items):
            input_ids[row, : len(item["input_ids"])] = torch.tensor(item["input_ids"])
        targets = {
            name: torch.tensor(
                [it["targets"][name] for it in items],
                dtype=torch.long if isinstance(items[0]["targets"][name], int) else torch.float32,
            )
            for name in items[0]["targets"]
        }
        return {"input_ids": input_ids, "lengths": lengths, "targets": targets}


def describe_columns() -> str:
    """Short summary of inputs and targets, for logs and docs."""
    return f"{len(FEATURES)} input columns -> {len(TARGET_COLUMNS)} targets ({TARGET_COLUMNS})"

### 3.3 Look at the data

[IBM HR Analytics Employee Attrition](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset):
1,470 fictional employees created by IBM data scientists. The split is 80/10/10,
stratified on attrition (16% of employees leave).

In [ ]:
paths = prepare_data(data_config)
train_frame = load_split(data_config, "train")
example = frame_to_records(train_frame)[0]
print(record_to_text(example), "\n")
print({c: example[c] for c in TARGET_COLUMNS})

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, col in zip(axes, TARGET_COLUMNS):
    if col == "MonthlyIncome":
        ax.hist(train_frame[col], bins=30)
    else:
        train_frame[col].value_counts().sort_index().plot.bar(ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Model

### 4.0 How the architecture works

```text
employee row (27 columns; the 4 targets are never inputs)
   │  record_to_text (section 3.1)
   ▼
"The employee is 41 years old, is female, ..., works overtime, ..."
   │  GPT-2 tokenizer + <eos>, right-padded with <|pad|>
   ▼
input_ids [B, T]   (T ≈ 165)          lengths [B]
   │  token embedding + position embedding
   ▼
[B, T, 768]
   │  12 × Block:  x = x + Attention(LayerNorm(x))   (causal: each token sees only the past)
   │               x = x + MLP(LayerNorm(x))          (768 → 3072 → 768)
   │  final LayerNorm
   ▼
[B, T, 768]
   │  take each row's last real token (<eos>, position lengths−1):
   │  it is the only token that has read the whole sentence, and padding comes after it
   ▼
[B, 768] ─┬─► Attrition head        [B, 2] → softmax → Stays / Leaves
          ├─► JobLevel head         [B, 5] → softmax → Level 1..5
          ├─► JobSatisfaction head  [B, 4] → softmax → Low..Very High
          └─► MonthlyIncome head    [B]    → exp(z·std + mean) → dollars
```

- **Why a sentence?** GPT-2 already knows what "works overtime" or "Research Director"
  mean. With only 1,176 training employees, that prior knowledge is the point.
- **Backbone:** GPT-2 written from scratch below, loaded with OpenAI's GPT-2 weights
  (the attention/MLP matrices are transposed: Hugging Face stores them as `Conv1D`).
  The token embedding table (38.6M params) is frozen; the rest is fine-tuned.
- **Heads:** `Linear(768, 768) → GELU → Dropout → Linear(768, k)`, one per task. They
  all read the same vector, so **one forward pass answers all four questions** and the
  four tasks train the backbone together.
- **Loss:** sum of cross-entropy (×3; attrition is class-weighted because only 16%
  leave) and MSE on standardised log(income).
- **Optimiser:** AdamW, 3e-5 for the pretrained backbone and 1e-3 for the new heads,
  warmup + cosine, gradient clipping, early stopping on the validation loss.

### 4.1 The code



*Same code as `src/hr_gpt/model.py`.*

HR-GPT: a GPT-2 style Transformer that reads an employee sentence and feeds 4 task heads.

```text
    "The employee is 41 years old, ..." <eos>
        -> token + position embeddings
        -> N causal Transformer blocks
        -> hidden state at <eos> (it has attended to the whole sentence)
        -> one small MLP head per task: Attrition, JobLevel, JobSatisfaction, MonthlyIncome

The backbone can start from the real GPT-2 (or DistilGPT-2) weights, which is what
makes this work on only ~1.2k training rows.
```

In [ ]:
INIT_STD = 0.02  # GPT-2 initialisation


class CausalSelfAttention(nn.Module):
    """Multi-head self-attention where each token sees only itself and earlier tokens."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_head = config.n_head
        self.dropout = config.dropout
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)  # q, k, v
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x: Tensor) -> Tensor:
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        # [B, T, C] -> [B, heads, T, C / heads]
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU(approximate="tanh")  # the GELU variant GPT-2 was trained with
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: Tensor) -> Tensor:
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    """Pre-norm Transformer block: x + attn(ln(x)), then x + mlp(ln(x))."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


class TaskHead(nn.Module):
    """[B, n_embd] -> [B, n_outputs]: Linear -> GELU -> Dropout -> Linear."""

    def __init__(self, n_embd: int, n_outputs: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(n_embd, n_outputs),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class HRGPT(nn.Module):
    def __init__(self, config: ModelConfig, tasks: dict[str, TaskSpec]):
        super().__init__()
        self.config = config.validate()
        self.tasks = dict(tasks)
        self.transformer = nn.ModuleDict(
            {
                "wte": nn.Embedding(config.vocab_size, config.n_embd),
                "wpe": nn.Embedding(config.block_size, config.n_embd),
                "drop": nn.Dropout(config.dropout),
                "h": nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                "ln_f": nn.LayerNorm(config.n_embd, bias=config.bias),
            }
        )
        self.heads = nn.ModuleDict(
            {
                name: TaskHead(config.n_embd, spec.n_outputs, config.head_dropout)
                for name, spec in tasks.items()
            }
        )
        self.apply(self._init_weights)
        # GPT-2: scale the residual projections by 1/sqrt(2 * n_layer)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=INIT_STD / math.sqrt(2 * config.n_layer))

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=INIT_STD)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=INIT_STD)

    # -- forward ------------------------------------------------------------

    def encode(self, input_ids: Tensor, lengths: Tensor) -> Tensor:
        """Token ids [B, T] -> one vector per sentence [B, n_embd], read at the last real token."""
        B, T = input_ids.shape
        if T > self.config.block_size:
            raise ValueError(f"sequence length {T} > block_size {self.config.block_size}")
        pos = torch.arange(T, device=input_ids.device)
        x = self.transformer.drop(self.transformer.wte(input_ids) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        last = (lengths - 1).to(input_ids.device)  # padding is at the end, after this token
        return x[torch.arange(B, device=input_ids.device), last]

    def forward(self, input_ids: Tensor, lengths: Tensor) -> dict[str, Tensor]:
        """Raw outputs per task: logits [B, n_classes] or a standardised value [B]."""
        h = self.encode(input_ids, lengths)
        out = {}
        for name, head in self.heads.items():
            y = head(h)
            out[name] = y if self.tasks[name].kind == CLASSIFICATION else y.squeeze(-1)
        return out

    def loss(
        self,
        outputs: dict[str, Tensor],
        targets: dict[str, Tensor],
        class_weights: dict[str, Tensor | None] | None = None,
    ) -> tuple[Tensor, dict[str, float]]:
        """Weighted sum of per-task losses: cross-entropy (classification), MSE (regression)."""
        class_weights = class_weights or {}
        total = outputs[next(iter(outputs))].new_zeros(())
        parts: dict[str, float] = {}
        for name, spec in self.tasks.items():
            y = targets[name].to(outputs[name].device)
            if spec.kind == CLASSIFICATION:
                w = class_weights.get(name)
                loss = F.cross_entropy(
                    outputs[name].float(), y, weight=None if w is None else w.to(y.device)
                )
            else:
                loss = F.mse_loss(outputs[name].float(), y)
            total = total + spec.loss_weight * loss
            parts[name] = loss.item()
        return total, parts

    # -- helpers ------------------------------------------------------------

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def param_groups(self, lr: float, head_lr: float, weight_decay: float) -> list[dict]:
        """AdamW groups: backbone vs heads (different LR), decay only on matrices."""
        groups: dict[tuple[bool, bool], list[nn.Parameter]] = {}
        for name, p in self.named_parameters():
            if not p.requires_grad:
                continue
            is_head = name.startswith("heads.")
            decay = p.dim() >= 2 and "wte" not in name and "wpe" not in name
            groups.setdefault((is_head, decay), []).append(p)
        return [
            {
                "params": params,
                "lr": head_lr if is_head else lr,
                "weight_decay": weight_decay if decay else 0.0,
            }
            for (is_head, decay), params in groups.items()
        ]


def config_for_pretrained(name: str, base: ModelConfig) -> ModelConfig:
    """Copy the architecture of a Hugging Face GPT-2 checkpoint into our config."""
    from transformers import AutoConfig

    hf = AutoConfig.from_pretrained(name)
    return ModelConfig(
        init=name,
        vocab_size=base.vocab_size,
        block_size=hf.n_positions,
        n_layer=hf.n_layer,
        n_head=hf.n_head,
        n_embd=hf.n_embd,
        dropout=base.dropout,
        head_dropout=base.head_dropout,
        bias=True,
    )


@torch.no_grad()
def load_pretrained_backbone(model: HRGPT, name: str) -> int:
    """Copy GPT-2 / DistilGPT-2 weights into the backbone. Returns the number of tensors copied.

    Hugging Face stores the attention/MLP matrices as Conv1D ([in, out]); ours are
    nn.Linear ([out, in]), so those are transposed. The extra <|pad|> embedding row
    gets the mean of GPT-2's embeddings. The task heads stay random.
    """
    from transformers import GPT2Model

    if name not in PRETRAINED:
        raise ValueError(f"unknown pretrained model {name!r}, expected one of {PRETRAINED}")
    hf = GPT2Model.from_pretrained(name).state_dict()
    ours = model.state_dict()
    transposed = (
        "attn.c_attn.weight",
        "attn.c_proj.weight",
        "mlp.c_fc.weight",
        "mlp.c_proj.weight",
    )
    copied = 0
    for key, value in hf.items():
        target = f"transformer.{key}"
        if target not in ours:
            continue  # e.g. the attention mask buffers of old transformers versions
        if key == "wte.weight":
            n = value.shape[0]
            ours[target][:n].copy_(value)
            ours[target][n:].copy_(value.mean(dim=0, keepdim=True))
        elif key.endswith(transposed):
            ours[target].copy_(value.t())
        else:
            ours[target].copy_(value)
        copied += 1
    expected = sum(1 for k in ours if k.startswith("transformer."))
    if copied != expected:
        raise RuntimeError(f"copied {copied} of {expected} backbone tensors from {name}")
    return copied


def build_model(
    config: ModelConfig, tasks: dict[str, TaskSpec], load_weights: bool = True
) -> HRGPT:
    """Create the model; with config.init = gpt2/distilgpt2, size and (optionally) weights come from it."""
    config = config.validate()
    if config.init in PRETRAINED:
        config = config_for_pretrained(config.init, config)
    model = HRGPT(config, tasks)
    if load_weights and config.init in PRETRAINED:
        load_pretrained_backbone(model, config.init)
    return model

### 4.2 Check the shapes on one real batch
Builds the model and pushes 4 real employees through it, printing each step's shape.

In [ ]:
tokenizer = build_tokenizer("gpt2")
demo_targets = TargetEncoder.fit(TASKS, train_frame)
demo_set = HRDataset(train_frame.iloc[:4], tokenizer, demo_targets, data_config.max_tokens)
batch = Collator(tokenizer.pad_token_id)([demo_set[i] for i in range(4)])
demo_model = build_model(replace(model_config, vocab_size=len(tokenizer)), TASKS).eval()
with torch.no_grad():
    pooled = demo_model.encode(batch["input_ids"], batch["lengths"])
    outputs = demo_model(batch["input_ids"], batch["lengths"])
print("input_ids        ", tuple(batch["input_ids"].shape), "  lengths", batch["lengths"].tolist())
print("pooled <eos> state", tuple(pooled.shape))
for name, out in outputs.items():
    print(f"{name:17s}", tuple(out.shape))
print(f"parameters: {demo_model.num_parameters() / 1e6:.1f}M")
del demo_model

### 4.3 Metrics and inference helpers

Accuracy alone is misleading with 84% stayers, so we also look at balanced accuracy, macro-F1, ROC-AUC, MAE and R².

*Same code as `src/hr_gpt/metrics.py`.*

Task metrics. Accuracy alone is misleading here (84% of employees stay), so we also report

```text
balanced accuracy and macro-F1 for every classifier, ROC-AUC for attrition, and MAE / R² for income.
```

In [ ]:
HEADLINE = {CLASSIFICATION: "balanced_accuracy", "regression": "r2"}


def classification_metrics(y_true: np.ndarray, probs: np.ndarray) -> dict[str, float]:
    """y_true [N] class ids, probs [N, C] -> accuracy, balanced accuracy, macro-F1 (+ AUC if C=2)."""
    pred = probs.argmax(axis=1)
    labels = list(range(probs.shape[1]))
    out = {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, labels=labels, average="macro", zero_division=0),
    }
    if probs.shape[1] == 2 and len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, probs[:, 1])
        out["f1_positive"] = f1_score(y_true, pred, pos_label=1, zero_division=0)
    return {k: float(v) for k, v in out.items()}


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """In the target's own units (e.g. dollars per month)."""
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(np.mean((y_true - y_pred) ** 2))),
        "r2": float(r2_score(y_true, y_pred)),
    }


def task_metrics(spec: TaskSpec, y_true: np.ndarray, prediction: np.ndarray) -> dict[str, float]:
    if spec.kind == CLASSIFICATION:
        return classification_metrics(y_true, prediction)
    return regression_metrics(y_true, prediction)


def headline(spec: TaskSpec, metrics: dict[str, float]) -> float:
    return metrics[HEADLINE[spec.kind]]

### 4.4 Predictor



*Same code as `src/hr_gpt/predict.py`.*

Load a trained checkpoint and predict all four tasks for one or many employees.

```text
Python:
    from hr_gpt.predict import Predictor
    predictor = Predictor.from_checkpoint("checkpoints/run/best.pt")
    predictor.predict(record)              # one employee (dict of the 27 input columns)

Command line:
    hr-predict --checkpoint checkpoints/run/best.pt --json employee.json
    hr-predict --checkpoint checkpoints/run/best.pt --test-row 0
```

In [ ]:
def pick_device(name: str = "auto") -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def run_model(
    model: HRGPT,
    ids: list[list[int]],
    pad_token_id: int,
    targets: TargetEncoder,
    batch_size: int = 32,
) -> dict[str, np.ndarray]:
    """Token ids of N sentences -> {task: probs [N, C] or values in real units [N]}."""
    model.eval()
    device = next(model.parameters()).device
    collate = Collator(pad_token_id)
    chunks: dict[str, list[np.ndarray]] = {name: [] for name in model.tasks}
    for start in range(0, len(ids), batch_size):
        batch = collate([{"input_ids": x, "targets": {}} for x in ids[start : start + batch_size]])
        outputs = model(batch["input_ids"].to(device), batch["lengths"].to(device))
        for name, out in outputs.items():
            if model.tasks[name].kind == CLASSIFICATION:
                value = torch.softmax(out.float(), dim=-1)
            else:
                value = targets.scales[name].decode(out.float())
            chunks[name].append(value.cpu().numpy())
    return {name: np.concatenate(parts) for name, parts in chunks.items()}


class Predictor:
    """A trained model + tokenizer + target scaling, ready for inference."""

    def __init__(
        self, model: HRGPT, tokenizer, targets: TargetEncoder, max_tokens: int, info: dict
    ):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.targets = targets
        self.max_tokens = max_tokens
        self.info = info  # epoch, val metrics, ... from the checkpoint

    @property
    def tasks(self) -> dict[str, TaskSpec]:
        return self.model.tasks

    @classmethod
    def from_checkpoint(cls, path: str | Path, device: str = "auto") -> "Predictor":
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(
                f"checkpoint not found: {path}. Train one first with `hr-train` (see README)."
            )
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
        tasks = tasks_from_dict(ckpt["tasks"])
        model = HRGPT(ModelConfig(**ckpt["model_config"]), tasks)
        model.load_state_dict(ckpt["model"])
        model.to(pick_device(device))
        tokenizer_dir = path.parent / ckpt["tokenizer_dir"]
        tokenizer = build_tokenizer(str(tokenizer_dir) if tokenizer_dir.exists() else "gpt2")
        targets = TargetEncoder.from_state_dict(tasks, ckpt["targets"])
        info = {k: ckpt.get(k) for k in ("epoch", "step", "val_loss", "val_metrics")}
        return cls(model, tokenizer, targets, ckpt["max_tokens"], info)

    def predict_many(self, records: list[dict[str, Any]]) -> dict[str, np.ndarray]:
        ids = [encode_text(self.tokenizer, record_to_text(r), self.max_tokens) for r in records]
        return run_model(self.model, ids, self.tokenizer.pad_token_id, self.targets)

    def predict(self, record: dict[str, Any]) -> dict[str, dict[str, Any]]:
        """One employee -> readable predictions for every task.

        Classification: {"label": "Leaves", "probabilities": {"Stays": 0.7, "Leaves": 0.3}}
        Regression:     {"value": 6123.4}
        """
        record = validate_record(record)
        raw = self.predict_many([record])
        result: dict[str, dict[str, Any]] = {}
        for name, spec in self.tasks.items():
            if spec.kind == CLASSIFICATION:
                probs = raw[name][0]
                result[name] = {
                    "label": spec.classes[int(probs.argmax())],
                    "probabilities": {c: float(p) for c, p in zip(spec.classes, probs)},
                }
            else:
                result[name] = {"value": float(raw[name][0])}
        return result


def actual_values(record: dict[str, Any], tasks: dict[str, TaskSpec]) -> dict[str, Any]:
    """The true targets of a dataset row, written like the predictions (for comparison)."""
    out = {}
    for name, spec in tasks.items():
        value = record.get(spec.column)
        if value is None:
            continue
        out[name] = spec.classes[spec.values.index(value)] if spec.kind == CLASSIFICATION else value
    return out


def format_prediction(
    result: dict[str, dict[str, Any]], actual: dict[str, Any] | None = None
) -> str:
    lines = []
    for name, res in result.items():
        if "label" in res:
            conf = res["probabilities"][res["label"]]
            text = f"{name:16s} {res['label']} ({conf:.0%})"
        else:
            text = f"{name:16s} ${res['value']:,.0f}"
        if actual and name in actual:
            truth = actual[name]
            text += f"   | actual: {truth if isinstance(truth, str) else f'${truth:,.0f}'}"
        lines.append(text)
    return "\n".join(lines)

## 5. Training



*Same code as `src/hr_gpt/train.py`.*

Training loop and the `hr-train` command.

```text
Each epoch: train on every employee once, then measure the validation split.
`last.pt` is written every epoch, `best.pt` whenever the validation loss improves.
Training stops early after `patience` epochs without improvement.

    hr-train                                   # GPT-2 small, default settings
    hr-train --init distilgpt2 --epochs 6      # smaller and ~2x faster
    hr-train --limit 64 --epochs 2 --init scratch --layers 2 --embd 128 --heads 4   # smoke test
```

In [ ]:
BEST_CHECKPOINT = "best.pt"


LAST_CHECKPOINT = "last.pt"


TOKENIZER_DIR = "tokenizer"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def lr_lambda(step: int, warmup_steps: int, total_steps: int) -> float:
    """LR multiplier: linear warmup to 1, then cosine decay to 0."""
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))


def autocast_for(device: torch.device, precision: str):
    """bf16 autocast on GPUs that support it (no loss scaling needed), else plain fp32."""
    use_bf16 = precision == "bf16" or (
        precision == "auto" and device.type == "cuda" and torch.cuda.is_bf16_supported()
    )
    if use_bf16 and device.type in ("cuda", "cpu"):
        return torch.autocast(device.type, dtype=torch.bfloat16)
    return nullcontext()


@torch.no_grad()
def evaluate(
    model: HRGPT,
    dataset: HRDataset,
    targets: TargetEncoder,
    pad_token_id: int,
    class_weights: dict[str, torch.Tensor | None],
    batch_size: int = 32,
) -> tuple[float, dict[str, dict[str, float]]]:
    """Validation loss (same weighting as training) and metrics per task."""
    model.eval()
    device = next(model.parameters()).device
    collate = Collator(pad_token_id)
    total, n = 0.0, 0
    for start in range(0, len(dataset), batch_size):
        batch = collate([dataset[i] for i in range(start, min(start + batch_size, len(dataset)))])
        outputs = model(batch["input_ids"].to(device), batch["lengths"].to(device))
        loss, _ = model.loss(outputs, batch["targets"], class_weights)
        total += loss.item() * len(batch["lengths"])
        n += len(batch["lengths"])
    preds = run_model(model, dataset.ids, pad_token_id, targets, batch_size)
    metrics = {}
    for name, spec in model.tasks.items():
        if spec.kind == "classification":
            y_true = np.array([t[name] for t in dataset.targets])
        else:
            y_true = np.array([r[spec.column] for r in dataset.records], dtype=float)
        metrics[name] = task_metrics(spec, y_true, preds[name])
    return total / max(1, n), metrics


def save_checkpoint(path: Path, model: HRGPT, extra: dict[str, Any]) -> None:
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), **extra}, tmp)
    tmp.replace(path)  # never leave a half-written checkpoint


def format_metrics(tasks: dict[str, TaskSpec], metrics: dict[str, dict[str, float]]) -> str:
    parts = []
    for name, spec in tasks.items():
        m = metrics[name]
        if spec.kind == "classification":
            extra = f" auc={m['roc_auc']:.3f}" if "roc_auc" in m else ""
            parts.append(f"{name}: bal_acc={m['balanced_accuracy']:.3f}{extra}")
        else:
            parts.append(f"{name}: mae=${m['mae']:,.0f} r2={m['r2']:.3f}")
    return " | ".join(parts)


def train(
    data_config: DataConfig,
    model_config: ModelConfig,
    train_config: TrainConfig,
    tasks: dict[str, TaskSpec] | None = None,
    tokenizer=None,
    model: HRGPT | None = None,
    on_epoch_end: Callable[[list[dict[str, Any]]], None] | None = None,
) -> dict[str, Any]:
    """Train and return a summary (best epoch, best val loss, history). `tokenizer`/`model`
    can be passed in (tests use tiny ones); otherwise GPT-2's tokenizer and `build_model` are used.
    `on_epoch_end(history)` is called after each epoch (the notebook draws its dashboard there).
    """
    train_config.validate()
    tasks = dict(tasks or TASKS)
    seed_everything(train_config.seed)
    device = pick_device(train_config.device)
    out = Path(train_config.output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # data
    train_frame = load_split(data_config, "train")
    if train_config.limit:
        train_frame = train_frame.iloc[: train_config.limit]
    val_frame = load_split(data_config, "val")
    tokenizer = tokenizer or build_tokenizer("gpt2")
    targets = TargetEncoder.fit(tasks, train_frame)
    class_weights = {name: targets.class_weights(name, train_frame) for name in tasks}
    train_set = HRDataset(train_frame, tokenizer, targets, data_config.max_tokens)
    val_set = HRDataset(val_frame, tokenizer, targets, data_config.max_tokens)
    pad_id = tokenizer.pad_token_id
    loader = DataLoader(
        train_set,
        batch_size=train_config.batch_size,
        shuffle=True,
        collate_fn=Collator(pad_id),
        num_workers=train_config.num_workers,
        generator=torch.Generator().manual_seed(train_config.seed),
    )
    print(describe_columns())
    print(f"train={len(train_set)} val={len(val_set)} employees | device={device}")

    # model + optimisation
    model_config.vocab_size = len(tokenizer)
    model = (model or build_model(model_config, tasks)).to(device)
    if train_config.freeze_embeddings:
        model.transformer.wte.weight.requires_grad_(False)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(
        f"model: init={model.config.init} layers={model.config.n_layer} "
        f"d={model.config.n_embd} params={model.num_parameters() / 1e6:.1f}M "
        f"(trainable {trainable / 1e6:.1f}M)"
    )
    optimizer = AdamW(
        model.param_groups(train_config.lr, train_config.head_lr, train_config.weight_decay)
    )
    total_steps = train_config.epochs * len(loader)
    warmup = max(1, int(train_config.warmup_fraction * total_steps))
    scheduler = LambdaLR(optimizer, lambda s: lr_lambda(s, warmup, total_steps))
    tokenizer.save_pretrained(out / TOKENIZER_DIR)
    autocast = autocast_for(device, train_config.precision)

    meta = {
        "model_config": to_jsonable(model.config),
        "tasks": to_jsonable(tasks),
        "targets": targets.state_dict(),
        "max_tokens": data_config.max_tokens,
        "tokenizer_dir": TOKENIZER_DIR,
        "data_config": to_jsonable(data_config),
        "train_config": to_jsonable(train_config),
    }
    history: list[dict[str, Any]] = []
    best_loss, best_epoch, bad_epochs, step = float("inf"), -1, 0, 0
    start = time.time()

    for epoch in range(1, train_config.epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for batch in loader:
            with autocast:
                outputs = model(batch["input_ids"].to(device), batch["lengths"].to(device))
            loss, parts = model.loss(outputs, batch["targets"], class_weights)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.grad_clip)
            optimizer.step()
            scheduler.step()
            step += 1
            running += loss.item() * len(batch["lengths"])
            seen += len(batch["lengths"])
            if step % train_config.log_every == 0:
                detail = " ".join(f"{k}={v:.3f}" for k, v in parts.items())
                print(
                    f"epoch {epoch} step {step}/{total_steps} loss={loss.item():.4f} "
                    f"({detail}) lr={scheduler.get_last_lr()[0]:.2e} {time.time() - start:.0f}s"
                )

        val_loss, val_metrics = evaluate(model, val_set, targets, pad_id, class_weights)
        row = {
            "epoch": epoch,
            "step": step,
            "train_loss": running / max(1, seen),
            "val_loss": val_loss,
            "val": val_metrics,
            "val_headline": {n: headline(s, val_metrics[n]) for n, s in tasks.items()},
            "seconds": round(time.time() - start, 1),
        }
        history.append(row)
        print(f"== epoch {epoch}: train_loss={row['train_loss']:.4f} val_loss={val_loss:.4f}")
        print(f"   {format_metrics(tasks, val_metrics)}")

        info = {"epoch": epoch, "step": step, "val_loss": val_loss, "val_metrics": val_metrics}
        save_checkpoint(out / LAST_CHECKPOINT, model, {**meta, **info})
        if val_loss < best_loss:
            best_loss, best_epoch, bad_epochs = val_loss, epoch, 0
            save_checkpoint(out / BEST_CHECKPOINT, model, {**meta, **info})
            print(f"   new best -> {out / BEST_CHECKPOINT}")
        else:
            bad_epochs += 1
        (out / "history.json").write_text(json.dumps(history, indent=2))
        if on_epoch_end:
            on_epoch_end(history)
        if train_config.patience and bad_epochs >= train_config.patience:
            print(f"stopping early: no improvement for {bad_epochs} epochs")
            break

    summary = {"best_epoch": best_epoch, "best_val_loss": best_loss, "history": history}
    print(f"done in {time.time() - start:.0f}s; best epoch {best_epoch} (val_loss={best_loss:.4f})")
    return summary

### 5.1 Train with a live dashboard

**What to expect.** Measured on the same test split, the bar to clear is the linear
baseline: attrition ROC-AUC 0.853, job level accuracy 76.2%, income MAE $1,388
(R² 0.825). Job satisfaction is at chance for every model (balanced accuracy ≈ 25%):
it is not predictable from these columns.

The plots refresh after each epoch: the losses on the left, then the main validation
metric of each task (balanced accuracy for the classifiers, R² for income).
Dashed lines are the "always guess the majority" level.

In [ ]:
CHANCE = {"Attrition": 0.5, "JobLevel": 0.2, "JobSatisfaction": 0.25, "MonthlyIncome": 0.0}


def dashboard(history):
    clear_output(wait=True)
    epochs = [h["epoch"] for h in history]
    _, axes = plt.subplots(1, 5, figsize=(20, 3.2))
    axes[0].plot(epochs, [h["train_loss"] for h in history], "o-", label="train")
    axes[0].plot(epochs, [h["val_loss"] for h in history], "o-", label="val")
    axes[0].set_title("loss")
    axes[0].legend()
    for ax, name in zip(axes[1:], TASKS):
        ax.plot(epochs, [h["val_headline"][name] for h in history], "o-")
        ax.axhline(CHANCE[name], ls="--", c="gray")
        ax.set_title(f"{name} ({HEADLINE[TASKS[name].kind]})")
    for ax in axes:
        ax.set_xlabel("epoch")
    plt.tight_layout()
    plt.show()
    last = history[-1]
    print(f"epoch {last['epoch']}: val_loss={last['val_loss']:.4f}  ({last['seconds']:.0f}s)")
    print(format_metrics(TASKS, last["val"]))


summary = train(data_config, model_config, train_config, on_epoch_end=dashboard)
print(f"best epoch: {summary['best_epoch']} (val_loss={summary['best_val_loss']:.4f})")

## 6. Test results vs baselines

The test split (147 employees) is used once, here, with the best checkpoint.

*Same code as `src/hr_gpt/evaluate.py`.*

`hr-evaluate`: score a checkpoint on the test split and compare it with two baselines.

```text
Baselines use the same 27 input columns as the model (never the other targets):
- trivial:  always the most frequent class / the median income
- linear:   logistic regression / ridge regression on one-hot + standardised columns

    hr-evaluate --checkpoint checkpoints/run/best.pt
    -> prints a table, writes test_results.json and test_results.md next to the checkpoint
```

In [ ]:
CATEGORICAL = [
    f.name for f in FEATURES if f.is_categorical and isinstance(next(iter(f.choices)), str)
]


NUMERIC = [f.name for f in FEATURES if f.name not in CATEGORICAL]


def true_values(spec: TaskSpec, frame: pd.DataFrame) -> np.ndarray:
    """Class ids (classification) or raw values (regression)."""
    if spec.kind == CLASSIFICATION:
        return frame[spec.column].map(lambda v: spec.values.index(v)).to_numpy()
    return frame[spec.column].astype(float).to_numpy()


def trivial_baseline(spec: TaskSpec, train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    y = true_values(spec, train)
    if spec.kind == CLASSIFICATION:
        prior = np.bincount(y, minlength=len(spec.classes)) / len(y)
        return np.tile(prior, (len(test), 1))  # argmax = majority class; AUC = 0.5
    return np.full(len(test), np.median(y))


def linear_baseline(spec: TaskSpec, train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    columns = ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
            ("num", StandardScaler(), NUMERIC),
        ]
    )
    y = true_values(spec, train)
    if spec.kind == CLASSIFICATION:
        clf = LogisticRegression(max_iter=5000, class_weight="balanced" if spec.balanced else None)
        model = make_pipeline(columns, clf).fit(train, y)
        return model.predict_proba(test)
    target = np.log(y) if spec.log_target else y
    model = make_pipeline(columns, Ridge(alpha=1.0)).fit(train, target)
    pred = model.predict(test)
    return np.exp(pred) if spec.log_target else pred


def evaluate_checkpoint(
    checkpoint: str | Path, data_config: DataConfig, split: str = "test"
) -> dict[str, Any]:
    train = load_split(data_config, "train")
    test = load_split(data_config, split)
    predictor = Predictor.from_checkpoint(checkpoint)
    model_preds = predictor.predict_many(frame_to_records(test))
    tasks = predictor.tasks or TASKS
    results: dict[str, Any] = {"split": split, "rows": len(test), "checkpoint": str(checkpoint)}
    for name, spec in tasks.items():
        y = true_values(spec, test)
        results[name] = {
            "HR-GPT": task_metrics(spec, y, model_preds[name]),
            "linear": task_metrics(spec, y, linear_baseline(spec, train, test)),
            "trivial": task_metrics(spec, y, trivial_baseline(spec, train, test)),
        }
    return results


ROWS = {
    CLASSIFICATION: [
        ("accuracy", "accuracy", "{:.1%}"),
        ("balanced_accuracy", "balanced acc.", "{:.1%}"),
        ("macro_f1", "macro-F1", "{:.3f}"),
        ("roc_auc", "ROC-AUC", "{:.3f}"),
    ],
    "regression": [
        ("mae", "MAE", "${:,.0f}"),
        ("rmse", "RMSE", "${:,.0f}"),
        ("r2", "R²", "{:.3f}"),
    ],
}


def to_markdown(results: dict[str, Any]) -> str:
    lines = [f"Test split: {results['rows']} employees never seen in training.\n"]
    for name, spec in TASKS.items():
        if name not in results:
            continue
        lines += [
            f"**{name}**\n",
            "| metric | HR-GPT | linear baseline | trivial baseline |",
            "|---|---:|---:|---:|",
        ]
        for key, label, fmt in ROWS[spec.kind]:
            if key not in results[name]["HR-GPT"]:
                continue
            cells = [
                fmt.format(results[name][m][key]) if key in results[name][m] else "-"
                for m in ("HR-GPT", "linear", "trivial")
            ]
            lines.append(f"| {label} | " + " | ".join(cells) + " |")
        lines.append("")
    return "\n".join(lines)

In [ ]:
results = evaluate_checkpoint(train_config.output_dir / BEST_CHECKPOINT, data_config)
(train_config.output_dir / "test_results.json").write_text(json.dumps(results, indent=2))
(train_config.output_dir / "test_results.md").write_text(to_markdown(results))
display(Markdown(to_markdown(results)))

## 7. Try the model

Load `best.pt` the way the repo's `hr-predict` and `hr-app` do, predict a few test
employees and compare with the truth. Then change one field and see what moves.

In [ ]:
predictor = Predictor.from_checkpoint(train_config.output_dir / BEST_CHECKPOINT)
test_rows = frame_to_records(load_split(data_config, "test"))
for row in test_rows[:3]:
    print(format_prediction(predictor.predict(row), actual_values(row, predictor.tasks)), "\n")

what_if = {**test_rows[0], "OverTime": "Yes", "YearsSinceLastPromotion": 8}
print("Same employee, with overtime and no promotion for 8 years:")
print(format_prediction(predictor.predict(what_if)))

## 8. Download the trained model

Zips `best.pt`, the tokenizer and the results. On your computer, with the repo installed:

```bash
unzip hr-gpt-model.zip -d checkpoints/run
hr-app --checkpoint checkpoints/run/best.pt
```

In [ ]:
bundle = BASE / "bundle"
shutil.rmtree(bundle, ignore_errors=True)
bundle.mkdir(parents=True)
for name in [BEST_CHECKPOINT, "history.json", "test_results.json", "test_results.md"]:
    shutil.copy(train_config.output_dir / name, bundle / name)
shutil.copytree(train_config.output_dir / TOKENIZER_DIR, bundle / TOKENIZER_DIR)
archive = shutil.make_archive(str(BASE / "hr-gpt-model"), "zip", bundle)
print("saved", archive, f"({Path(archive).stat().st_size / 1e6:.0f} MB)")
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Kaggle: find it under Output -> hr-gpt/hr-gpt-model.zip")